# Segmentation model 1

## Улучшения относительно baseline

### Loss функция
Используется **комбинированная функция потерь**:
$$
loss = 0.25 \cdot BCE_{WithLogitsLoss} + 0.75 \cdot DiceLoss
$$
→ Лучшая сходимость по сравнению с чистым Dice.

### Аугментации
Расширенный набор аугментаций на обучении (Albumentations):
- Horizontal / Vertical Flip, RandomRotate90
- ShiftScaleRotate, RandomBrightnessContrast, HueSaturationValue
- GaussNoise + CoarseDropout

### Архитектура модели
- **Модель**: `UnetPlusPlus` (одна из самых сильных для бинарной сегментации)
- **Энкодер**: `efficientnet-b3` (pretrained на ImageNet)
- Входной размер: **512×512**

### Кросс-валидация и обучение
- 5-fold Stratified KFold
- Каждая модель сохраняется (`best.pth` + `history.csv`)
- Автоматический бэкап лучших чекпоинтов и истории **на Google Drive**

### Оптимизации скорости и памяти
- `torch.backends.cudnn.benchmark + TF32 matmul`
- `torch.compile(mode="reduce-overhead")` (и на обучении, и на инференсе)
- Automatic Mixed Precision (AMP)
- DataLoader: 8 workers + prefetch_factor + persistent_workers

### Инференс
- **Ensemble** из 5 лучших моделей (по одному с каждого фолда)
- Test-Time Augmentation (TTA): оригинал + horizontal flip + vertical flip
- Автоматический бэкап `submission.csv` на Google Drive

### Еще идеи
- Soft threshold
- SWA


# Segmentation model 1

## Init

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
import os
from google.colab import userdata

# Получаем учетные данные из менеджера секретов Colab
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Устанавливаем переменные окружения для Kaggle
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key
os.environ['KAGGLEHUB_CACHE'] = '/'

# Создаем директорию .kaggle, если ее нет
!mkdir -p ~/.kaggle

# Создаем файл kaggle.json с учетными данными
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

# Устанавливаем правильные права доступа для файла kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Теперь вы можете использовать kaggle CLI или kagglehub.login()
# kagglehub.login() не всегда нужен, если kaggle.json настроен правильно
print("Kaggle API credentials configured successfully!")

Kaggle API credentials configured successfully!


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dl_lab_3_product_segmentation_path = kagglehub.competition_download('dl-lab-3-product-segmentation')
# packagemanager_pm_113281741_at_03_23_2026_18_04_38_path = kagglehub.notebook_output_download('packagemanager/pm-113281741-at-03-23-2026-18-04-38')

print('Data source import complete.')


100%|██████████| 191M/191M [00:03<00:00, 62.9MB/s]

Extracting files...


Data source import complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Путь к папке на Google Drive
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/segmentation_checkpoints")
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/segmentation_checkpoints


In [ ]:
pip install segmentation-models-pytorch timm opencv-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.8 MB/s eta 0:00:00


In [ ]:
pip uninstall -y torch torchvision

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128


In [ ]:
pip install --no-cache-dir torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 76.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 267.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 182.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 93.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 237.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 243.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 214.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 151.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 178.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 29.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 301.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 955.5/

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

## Preparation / train

### GPU speed optimization

In [ ]:
import torch

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

print(" CUDA optimizations enabled:")
print(" cudnn.benchmark = True")
print(" TF32 matmul enabled")
print(" high precision matmul")

 CUDA optimizations enabled:
 cudnn.benchmark = True
 TF32 matmul enabled
 high precision matmul


### CONFIG

In [ ]:
DATA_ROOT = Path(
    r"/competitions/dl-lab-3-product-segmentation/train"
)
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/segmentation_checkpoints")
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Флаг: копировать ли на Drive (можно отключить для быстрых тестов)
BACKUP_TO_DRIVE = True

IMG_SIZE = 512
BATCH_SIZE = 12 # тут смотреть по GPU mem
NUM_EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2

# === SPEED UPDATES ===
NUM_WORKERS = 2                    # было 4
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = True

SEED = 42
THRESHOLD = 0.5

MODEL_NAME = "UnetPlusPlus"
ENCODER_NAME = "efficientnet-b3" # или efficientnet-b0 для экономии времени
ENCODER_WEIGHTS = "imagenet"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import shutil

def backup_checkpoint_to_drive(local_path: Path, drive_path: Path, fold: int, epoch: int, is_best: bool = False):
    if not BACKUP_TO_DRIVE or not is_best:
        return

    try:
        drive_fold_dir = drive_path / f"fold_{fold}"
        drive_fold_dir.mkdir(parents=True, exist_ok=True)

        drive_file = drive_fold_dir / "best.pth"
        shutil.copy2(local_path, drive_file)

        meta_file = drive_fold_dir / f"checkpoint_metadata.txt"
        with open(meta_file, 'a') as f:
            f.write(f"Epoch {epoch}: BEST - {local_path.name}\n")

        print(f"  Backed up BEST model to Drive: {drive_file}")
    except Exception as e:
        print(f"  Failed to backup to Drive: {e}")

### Utils

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def dice_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()


def iou_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


### Loss (нейронка говорит, что если добавить BCE, будет лучше сходиться)

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

dice_loss = smp.losses.DiceLoss(
    mode=smp.losses.BINARY_MODE,
    from_logits=True
)

def combined_loss(logits, targets):
    bce = bce_loss(logits, targets)
    dice = dice_loss(logits, targets)
    return 0.25 * bce + 0.75 * dice

### Augmentations

In [ ]:
import albumentations as A

def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.Affine(
            scale=(0.9, 1.1),
            translate_percent=(-0.05, 0.05),
            rotate=(-20, 20),
            p=0.5
        ),

        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(p=0.3),
        A.GaussNoise(p=0.2),

        A.CoarseDropout(
            max_holes=8,
            hole_height_range=(1, img_size // 20),
            hole_width_range=(1, img_size // 20),
            p=0.3
        ),
    ])


def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
    ])

### Dataset

In [ ]:
class BinarySegDataset(Dataset):
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int = 352,
        encoder_name: str = "resnet34",
        encoder_weights: str | None = "imagenet",
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.transforms = transforms

        self.preprocess_input = None
        if encoder_weights is not None:
            self.preprocess_input = get_preprocessing_fn(
                encoder_name, pretrained=encoder_weights
            )

        self.samples = []
        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            image_path = find_image_for_stem(self.images_dir, stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise RuntimeError(
                f"No paired image/mask samples found in {self.images_dir} and {self.masks_dir}"
            )

        print(f"Found {len(self.samples)} paired samples")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]

        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise RuntimeError(f"Cannot read image: {image_path}")

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {mask_path}")

        if self.transforms is not None:
            augmented = self.transforms(image=image_rgb, mask=mask)
            image_rgb = augmented["image"]
            mask = augmented["mask"]
        else:
            image_rgb = cv2.resize(image_rgb, (self.img_size, self.img_size))
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        image_rgb = image_rgb.astype(np.float32)

        if self.preprocess_input is not None:
            image_rgb = self.preprocess_input(image_rgb)
        else:
            image_rgb = image_rgb / 255.0

        mask = (mask > 0).astype(np.float32)

        image = torch.from_numpy(image_rgb.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask[None, ...]).float()

        return image, mask

### Model

In [ ]:
def build_model() -> nn.Module:
    if MODEL_NAME == "Unet":
        model = smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "FPN":
        model = smp.FPN(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")
    return model


### TRAIN / VAL LOOPS

In [ ]:
from tqdm import tqdm


scaler = torch.cuda.amp.GradScaler()

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    pbar = tqdm(loader, desc="Training", leave=False)


    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = loss_fn(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits.detach(), masks)
        running_iou += iou_score_from_logits(logits.detach(), masks)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }


@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    pbar = tqdm(loader, desc="Validation", leave=False)


    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = loss_fn(logits, masks)

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits, masks)
        running_iou += iou_score_from_logits(logits, masks)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

/tmp/ipykernel_2904/669458370.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


### main

In [ ]:
seed_everything(SEED)

from sklearn.model_selection import KFold

N_FOLDS = 5

full_dataset = BinarySegDataset(
    images_dir=IMAGES_DIR,
    masks_dir=MASKS_DIR,
    img_size=IMG_SIZE,
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    transforms=None,
)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

all_folds_dice = []

train_transforms = get_train_transforms(IMG_SIZE)
val_transforms = get_val_transforms(IMG_SIZE)

for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset)):
    print(f"\n===== FOLD {fold+1}/{N_FOLDS} =====")

    train_dataset = torch.utils.data.Subset(
        BinarySegDataset(
            IMAGES_DIR,
            MASKS_DIR,
            IMG_SIZE,
            ENCODER_NAME,
            ENCODER_WEIGHTS,
            transforms=train_transforms,
        ),
        train_idx,
    )

    val_dataset = torch.utils.data.Subset(
        BinarySegDataset(
            IMAGES_DIR,
            MASKS_DIR,
            IMG_SIZE,
            ENCODER_NAME,
            ENCODER_WEIGHTS,
            transforms=val_transforms,
        ),
        val_idx,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=PREFETCH_FACTOR,
        persistent_workers=PERSISTENT_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=PREFETCH_FACTOR,
        persistent_workers=PERSISTENT_WORKERS
    )

    model = build_model().to(DEVICE)

    loss_fn = combined_loss

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS
    )

    best_val_dice = -1.0
    history = []

    SAVE_DIR_FOLD = SAVE_DIR / f"fold_{fold}"
    SAVE_DIR_FOLD.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, NUM_EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_metrics = validate_one_epoch(model, val_loader, loss_fn, DEVICE)

        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_dice": train_metrics["dice"],
            "train_iou": train_metrics["iou"],
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "val_iou": val_metrics["iou"],
        }
        history.append(row)

        print(
            f"Fold {fold+1} | Epoch {epoch:03d}/{NUM_EPOCHS} | "
            f"train_loss={row['train_loss']:.4f} train_dice={row['train_dice']:.4f} | "
            f"val_loss={row['val_loss']:.4f} val_dice={row['val_dice']:.4f}"
        )

        # Сохраняем чекпоинт
        checkpoint_data = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_dice": row["val_dice"],
            "config": {
                "MODEL_NAME": MODEL_NAME,
                "ENCODER_NAME": ENCODER_NAME,
                "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                "IMG_SIZE": IMG_SIZE,
            },
        }

        # Сохраняем локально last.pth
        last_path = SAVE_DIR_FOLD / "last.pth"
        torch.save(checkpoint_data, last_path)
        # backup_checkpoint_to_drive(last_path, DRIVE_CHECKPOINT_DIR, fold, epoch, is_best=False)

        if row["val_dice"] > best_val_dice:
            best_val_dice = row["val_dice"]

            # Сохраняем локально best.pth
            best_path = SAVE_DIR_FOLD / "best.pth"
            torch.save(checkpoint_data, best_path)
            backup_checkpoint_to_drive(best_path, DRIVE_CHECKPOINT_DIR, fold, epoch, is_best=True)

            print(f"Fold {fold+1}: saved best model (dice={best_val_dice:.4f})")

    import pandas as pd
    history_df = pd.DataFrame(history)
    history_df.to_csv(SAVE_DIR_FOLD / "history.csv", index=False)

    # Сохраняем историю на Drive
    if BACKUP_TO_DRIVE:
        try:
            drive_fold_dir = DRIVE_CHECKPOINT_DIR / f"fold_{fold}"
            drive_fold_dir.mkdir(parents=True, exist_ok=True)
            history_df.to_csv(drive_fold_dir / "history.csv", index=False)
            print(f"  Backed up history to Drive: {drive_fold_dir / 'history.csv'}")
        except Exception as e:
            print(f"  Failed to backup history to Drive: {e}")

    print(f"Fold {fold+1} finished. Best dice={best_val_dice:.4f}")

    all_folds_dice.append(best_val_dice)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


print("\n===== CROSS-VALIDATION RESULT =====")
print(f"Mean Dice: {np.mean(all_folds_dice):.4f}")
print(f"Std Dice:  {np.std(all_folds_dice):.4f}")

Found 2000 paired samples

===== FOLD 1/5 =====
Found 2000 paired samples
Found 2000 paired samples


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_2904/1114994549.py:24: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Training:   0%|          | 0/160 [00:00<?, ?it/s]/tmp/ipykernel_2904/669458370.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Validation:   0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipykernel_2904/669458370.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Fold 1 | Epoch 001/20 | train_loss=0.2654 train_dice=0.6967 | val_loss=0.1614 val_dice=0.7929
  Backed up BEST model to Drive: /content/drive/MyDrive/segmentation_checkpoints/fold_0/best.pth
Fold 1: saved best model (dice=0.7929)


...

Fold 5 | Epoch 016/20 | train_loss=0.0757 train_dice=0.8870 | val_loss=0.0806 val_dice=0.8956
  Backed up BEST model to Drive: /content/drive/MyDrive/segmentation_checkpoints/fold_4/best.pth
Fold 5: saved best model (dice=0.8956)


Fold 5 | Epoch 017/20 | train_loss=0.0730 train_dice=0.8888 | val_loss=0.0798 val_dice=0.8971
  Backed up BEST model to Drive: /content/drive/MyDrive/segmentation_checkpoints/fold_4/best.pth
Fold 5: saved best model (dice=0.8971)


Fold 5 | Epoch 018/20 | train_loss=0.0709 train_dice=0.8953 | val_loss=0.0801 val_dice=0.8958


Fold 5 | Epoch 019/20 | train_loss=0.0692 train_dice=0.8957 | val_loss=0.0792 val_dice=0.8961


Fold 5 | Epoch 020/20 | train_loss=0.0692 train_dice=0.8945 | val_loss=0.0792 val_dice=0.8960
  Backed up history to Drive: /content/drive/MyDrive/segmentation_checkpoints/fold_4/history.csv
Fold 5 finished. Best dice=0.8971

===== CROSS-VALIDATION RESULT =====
Mean Dice: 0.8940
Std Dice:  0.0057


## Submission formation

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn


In [ ]:
def load_checkpoint_from_drive(fold: int, checkpoint_type: str = "best"):
    """
    Загружает чекпоинт с Google Drive
    checkpoint_type: "best" или "last"
    """
    drive_fold_dir = DRIVE_CHECKPOINT_DIR / f"fold_{fold}"
    checkpoint_path = drive_fold_dir / f"{checkpoint_type}.pth"

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    print(f"Loading from Drive: {checkpoint_path}")
    return torch.load(checkpoint_path, map_location="cpu")

### Config

In [ ]:
TEST_IMAGES_DIR = Path(
    r"/competitions/dl-lab-3-product-segmentation/test_images"
)
OUTPUT_CSV = "submission.csv"

# путь к вашему чекпоинту после обучения
# CHECKPOINT_PATH = Path(r"./seg_train_runs/best.pth")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


### HELPERS

In [ ]:
def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)


def build_model(model_name: str, encoder_name: str, encoder_weights=None):
    if model_name == "Unet":
        model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "FPN":
        model = smp.FPN(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {model_name}")
    return model




def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))


### Load checkpoint



In [ ]:
models = []

for fold in range(N_FOLDS):
    ckpt_path = SAVE_DIR / f"fold_{fold}/best.pth"
    print(f"Loading: {ckpt_path}")

    checkpoint = torch.load(ckpt_path, map_location="cpu")

    if "config" not in checkpoint:
        raise ValueError(f"No config in checkpoint: {ckpt_path}")

    cfg = checkpoint["config"]

    model = build_model(
        model_name=cfg["MODEL_NAME"],
        encoder_name=cfg["ENCODER_NAME"],
        encoder_weights=None,
    )

    state_dict = checkpoint["model_state_dict"]
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith("_orig_mod."):
            new_key = key[10:]
        else:
            new_key = key
        new_state_dict[new_key] = value

    model.load_state_dict(new_state_dict)

    model.to(DEVICE)
    model.eval()

    if torch.__version__ >= (2, 0) and hasattr(torch, 'compile'):
        print(f"   → Compiling model fold {fold} (inference speedup)")
        model = torch.compile(
            model,
            mode="reduce-overhead",
            fullgraph=False,
            dynamic=False
        )
    models.append(model)

Loading: seg_train_runs/fold_0/best.pth
   → Compiling model fold 0 (inference speedup)
Loading: seg_train_runs/fold_1/best.pth
   → Compiling model fold 1 (inference speedup)
Loading: seg_train_runs/fold_2/best.pth
   → Compiling model fold 2 (inference speedup)
Loading: seg_train_runs/fold_3/best.pth
   → Compiling model fold 3 (inference speedup)
Loading: seg_train_runs/fold_4/best.pth
   → Compiling model fold 4 (inference speedup)


### INFERENCE

In [ ]:
preprocess_input = get_preprocessing_fn(ENCODER_NAME, pretrained=ENCODER_WEIGHTS)

image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])

if not image_paths:
    raise FileNotFoundError(f"No images found in: {TEST_IMAGES_DIR}")

print(f"Found {len(image_paths)} test images")

rows = []

with torch.no_grad():
    for i, img_path in enumerate(image_paths, 1):
        img_bgr = cv2_imread_unicode(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            print(f"[skip] cannot read: {img_path}")
            continue

        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        inp = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        inp = inp.astype(np.float32)

        if preprocess_input is not None:
            inp = preprocess_input(inp)
        else:
            inp = inp / 255.0

        inp = torch.from_numpy(inp.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        preds = []
        for model in models:
            tta_preds = []
            # original
            logits = model(inp)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            tta_preds.append(probs)
            # horizontal flip
            inp_flip = torch.flip(inp, dims=[3])
            logits = model(inp_flip)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            probs = np.flip(probs, axis=1)
            tta_preds.append(probs)
            # vertical flip
            inp_flip = torch.flip(inp, dims=[2])
            logits = model(inp_flip)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            probs = np.flip(probs, axis=0)
            tta_preds.append(probs)

            tta_pred = np.mean(tta_preds, axis=0)
            preds.append(tta_pred)

        pred = np.mean(preds, axis=0)
        if pred.shape != (H, W):
            pred = cv2.resize(pred.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)

        mask = (pred > THRESHOLD).astype(np.uint8)
        rows.append({"ImageId": img_path.name, "mask": serialize_mask(mask)})

        if i % 100 == 0 or i == len(image_paths):
            print(f"Processed {i}/{len(image_paths)}")

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nSaved local submission to: {OUTPUT_CSV}")

if BACKUP_TO_DRIVE:
    try:
        drive_submission_path = DRIVE_CHECKPOINT_DIR / "submission.csv"
        shutil.copy2(OUTPUT_CSV, drive_submission_path)
        print(f"Backed up submission to Drive: {drive_submission_path}")
    except Exception as e:
        print(f"Failed to backup submission to Drive: {e}")

display(submission_df.head())

Found 2000 test images
Processed 100/2000
Processed 200/2000
Processed 300/2000
Processed 400/2000
Processed 500/2000
Processed 600/2000
Processed 700/2000
Processed 800/2000
Processed 900/2000
Processed 1000/2000
Processed 1100/2000
Processed 1200/2000
Processed 1300/2000
Processed 1400/2000
Processed 1500/2000
Processed 1600/2000
Processed 1700/2000
Processed 1800/2000
Processed 1900/2000
Processed 2000/2000

Saved local submission to: submission.csv
Backed up submission to Drive: /content/drive/MyDrive/segmentation_checkpoints/submission.csv


,ImageId,mask
0,10.107.215.111_20260119142748_a0be018c-d498-4b...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
1,10.107.215.111_20260119203335_58a6ef5c-c360-4e...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
2,10.107.215.111_20260119203341_bb74ac11-3f14-4a...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
3,10.107.215.111_20260122115937_db9b4af5-0f1e-42...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
4,10.107.215.111_20260124135952_6fd575ec-50e8-4f...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
